# Solutions — Glue-Style Jobs: Parameters, Metrics, Bookmarks, a Complete Job

Exercise notebook: `11_python_for_aws/02_glue_style_jobs.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
S3_ROOT = LAB / "datasets/s3"        # local stand-in for S3: <root>/<bucket>/<key>
for p in sorted(S3_ROOT.rglob("*"))[:12]:
    print(p.relative_to(S3_ROOT))

## Exercise 1 — 🟢 Easy · getResolvedOptions

**Working solution**

In [ ]:
def get_resolved_options(argv, names):
    wanted, found = set(names), {}
    i = 0
    while i < len(argv):
        token = argv[i]
        if token.startswith("--"):
            name, sep, value = token[2:].partition("=")
            if not sep:
                value = argv[i + 1] if i + 1 < len(argv) else None
                i += 1
            if name in wanted and value is not None:
                found[name] = value
        i += 1
    missing = [n for n in names if n not in found]
    if missing:
        raise ValueError(f"missing required job arguments: {missing}")
    return {n: found[n] for n in names}

**Explanation**

Glue/Spark pass a noisy argv (`--job-bookmark-option`, `--TempDir`, ...); a resolver that extracts only what *your* job declares keeps jobs portable between Glue and local runs.

**Complexity:** O(len(argv)).

**Common mistakes**

- Using argparse with strict parsing on Glue's argv → fails on unknown internal args (use `parse_known_args`).

## Exercise 2 — 🟡 Medium · A proper CLI for local runs

**Working solution**

In [ ]:
import argparse
from datetime import date

def _s3_uri(value):
    if not value.startswith("s3://") or len(value) <= 5:
        raise argparse.ArgumentTypeError(f"not an s3 uri: {value!r}")
    return value

def _iso_date(value):
    try:
        return date.fromisoformat(value)
    except ValueError:
        raise argparse.ArgumentTypeError(f"expected YYYY-MM-DD, got {value!r}") from None

def _positive_int(value):
    n = int(value)
    if n < 1:
        raise argparse.ArgumentTypeError("must be >= 1")
    return n

def build_parser():
    p = argparse.ArgumentParser(prog="daily_sales_job", description="Validate and publish daily sales partitions.")
    p.add_argument("--config", required=True, type=_s3_uri, help="s3:// URI of the job config JSON")
    p.add_argument("--ingest-date", type=_iso_date, default=None, help="partition to process (default: all new partitions)")
    p.add_argument("--env", choices=["dev", "qa", "prod"], default="dev")
    p.add_argument("--batch-size", type=_positive_int, default=500)
    p.add_argument("--dry-run", action="store_true", help="validate only, write nothing")
    p.add_argument("--log-level", type=str.upper, default="INFO", choices=["DEBUG", "INFO", "WARNING", "ERROR"])
    return p

**Explanation**

`type=` functions convert *and* validate; `ArgumentTypeError` produces a clean usage error. Dashes become underscores on the namespace (`args.ingest_date`). `str.upper` as a `type`
normalizes before `choices` is checked — functions as values again.

**Complexity:** O(args).

**Common mistakes**

- Validating dates later in the job instead of at the CLI boundary.
- `type=bool` for flags (`bool('False')` is True) — use `action='store_true'`.

## Exercise 3 — 🟡 Medium · CloudWatch Embedded Metric Format

**Working solution**

In [ ]:
VALID_UNITS = {"Count", "Seconds", "Milliseconds", "Bytes", "Percent", "None"}

def emf_record(namespace, metrics, dimensions, timestamp_ms):
    if len(dimensions) > 30:
        raise ValueError("at most 30 dimensions")
    clash = set(metrics) & set(dimensions)
    if clash:
        raise ValueError(f"names used as both metric and dimension: {sorted(clash)}")
    for name, (_, unit) in metrics.items():
        if unit not in VALID_UNITS:
            raise ValueError(f"invalid unit {unit!r} for {name}")
    return {
        "_aws": {"Timestamp": timestamp_ms, "CloudWatchMetrics": [{
            "Namespace": namespace, "Dimensions": [list(dimensions)],
            "Metrics": [{"Name": n, "Unit": u} for n, (_, u) in metrics.items()]}]},
        **dimensions,
        **{n: v for n, (v, _) in metrics.items()},
    }

**Explanation**

One JSON log line both documents the run *and* creates metrics/alarms (rows rejected, duration) without calling the CloudWatch API. Dict unpacking assembles the flat top-level fields.

**Complexity:** O(metrics + dimensions).

**Common mistakes**

- Calling `put_metric_data` per row (slow, throttled, costly).

## Exercise 4 — 🟠 Hard · Job bookmarks over S3 objects

**Working solution**

In [ ]:
import json
import os
import tempfile
from pathlib import Path

class JobBookmark:
    def __init__(self, root, job_name):
        self.path = Path(root, "lab-processed-bucket", "_bookmarks", f"{job_name}.json")
        state = json.loads(self.path.read_text(encoding="utf-8")) if self.path.exists() else {}
        self.objects = dict(state.get("objects", {}))
        self.runs = state.get("runs", 0)
        self._staged = {}

    def filter_new(self, bucket, objects):
        return [o for o in objects if self.objects.get(f"{bucket}/{o['Key']}") != o["ETag"]]

    def mark(self, bucket, objects):
        for o in objects:
            self._staged[f"{bucket}/{o['Key']}"] = o["ETag"]

    def commit(self):
        self.objects.update(self._staged)
        self._staged = {}
        self.runs += 1
        self.path.parent.mkdir(parents=True, exist_ok=True)
        fd, tmp = tempfile.mkstemp(dir=self.path.parent, suffix=".tmp")
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump({"objects": self.objects, "runs": self.runs}, f, indent=2, sort_keys=True)
        os.replace(tmp, self.path)

    def reset(self):
        self.objects, self._staged, self.runs = {}, {}, 0
        self.path.unlink(missing_ok=True)

**Explanation**

Bookmarks are a checkpoint keyed by object identity (key + ETag), committed only after the job's outputs are written. Staging marks in memory gives 'commit on success' semantics:
a crash before `commit()` leaves the old bookmark, so the next run picks the objects up again.

**Complexity:** O(objects).

**Common mistakes**

- Bookmarking by key only — a corrected re-upload with the same key would be ignored.
- Committing at the start of the run.

## Challenge — 🔴 Very Hard · A complete Glue-style job

**Working solution**

In [ ]:
import csv
import hashlib
import io
import json
import shutil
import time
from pathlib import Path

def _resolve(argv, required, optional=()):
    found, i = {}, 0
    while i < len(argv):
        if argv[i].startswith("--"):
            name, sep, value = argv[i][2:].partition("=")
            if not sep and i + 1 < len(argv):
                value, i = argv[i + 1], i + 1
            found[name] = value
        i += 1
    missing = [n for n in required if n not in found]
    if missing:
        raise ValueError(f"missing required job arguments: {missing}")
    return {n: found.get(n) for n in [*required, *optional]}

def _local(root, uri):
    bucket, _, key = uri[5:].partition("/")
    return Path(root, bucket, *[p for p in key.split("/") if p])

def _bookmark_path(root, job):
    return Path(root, "lab-processed-bucket", "_bookmarks", f"{job}.json")

def run_sales_job(argv, root, now_ms=None):
    args = _resolve(argv, ["JOB_NAME", "config_uri"], ["ingest_date"])
    job, root = args["JOB_NAME"], Path(root)
    now_ms = now_ms if now_ms is not None else int(time.time() * 1000)
    cfg = json.loads(_local(root, args["config_uri"]).read_text(encoding="utf-8"))
    src_dir, pkey = _local(root, cfg["source"]), cfg["partition_key"]
    allowed = {s.upper() for s in cfg["allowed_status"]}

    objects = {}
    for f in sorted(src_dir.rglob("*.csv")):
        part = next((p.split("=", 1)[1] for p in f.relative_to(src_dir).parts if p.startswith(pkey + "=")), None)
        if part is not None:
            objects.setdefault(part, []).append((f, hashlib.md5(f.read_bytes()).hexdigest()))

    bm_path = _bookmark_path(root, job)
    bookmark = json.loads(bm_path.read_text()) if bm_path.exists() else {}
    if args["ingest_date"]:
        selected = {args["ingest_date"]: objects.get(args["ingest_date"], [])}
    else:
        selected = {p: objs for p, objs in objects.items() if any(bookmark.get(str(f.relative_to(root).as_posix())) != md5 for f, md5 in objs)}

    summary = {"job": job, "partitions": {}, "status": "SUCCEEDED"}
    for part, objs in sorted(selected.items()):
        stats = {"read": 0, "valid": 0, "rejected": 0, "duplicates": 0}
        best, rejects = {}, []
        for f, _ in objs:
            for row in csv.DictReader(io.StringIO(f.read_text(encoding="utf-8-sig"))):
                stats["read"] += 1
                try:
                    if not row.get("order_id"):
                        raise ValueError("missing order_id")
                    qty, price = int(row["quantity"]), float(row["unit_price"])
                    status = row["status"].strip().upper()
                    if qty < cfg["min_quantity"] or price < 0 or status not in allowed:
                        raise ValueError("rule violation")
                except (ValueError, KeyError, AttributeError) as exc:
                    rejects.append({**row, "_error": str(exc), "_source": f.name})
                    continue
                rec = {**row, "quantity": qty, "unit_price": price, "status": status}
                prev = best.get(rec["order_id"])
                if prev is not None:
                    stats["duplicates"] += 1
                    if prev["updated_at"] >= rec["updated_at"]:
                        continue
                best[rec["order_id"]] = rec
        stats["valid"], stats["rejected"] = len(best), len(rejects)
        for base, rows in ((_local(root, cfg["target"]), [best[k] for k in sorted(best)]), (_local(root, cfg["rejected"]), rejects)):
            out_dir = base / f"{pkey}={part}"
            shutil.rmtree(out_dir, ignore_errors=True)
            out_dir.mkdir(parents=True)
            (out_dir / "part-00000.jsonl").write_text("".join(json.dumps(r, sort_keys=True) + "\n" for r in rows), encoding="utf-8")
        summary["partitions"][part] = stats
        for f, md5 in objs:
            bookmark[f.relative_to(root).as_posix()] = md5

    bm_path.parent.mkdir(parents=True, exist_ok=True)
    bm_path.write_text(json.dumps(bookmark, indent=2, sort_keys=True), encoding="utf-8")
    totals = {k: sum(p[k] for p in summary["partitions"].values()) for k in ("read", "valid", "rejected")}
    summary["metrics"] = {"_aws": {"Timestamp": now_ms, "CloudWatchMetrics": [{"Namespace": "Lab/Glue", "Dimensions": [["JobName"]],
                          "Metrics": [{"Name": "RowsRead", "Unit": "Count"}, {"Name": "RowsValid", "Unit": "Count"}, {"Name": "RowsRejected", "Unit": "Count"}]}]},
                          "JobName": job, "RowsRead": totals["read"], "RowsValid": totals["valid"], "RowsRejected": totals["rejected"]}
    runs_dir = _local(root, cfg["target"]) / "_runs"
    runs_dir.mkdir(parents=True, exist_ok=True)
    (runs_dir / f"{job}-{now_ms}.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return summary

**Explanation**

This is the anatomy of a real Glue/EMR batch job: arguments → config → discover inputs → select work (bookmark or explicit partition) → per-partition validate/dedupe/overwrite →
commit progress → emit metrics and a run record. Swapping `Path` I/O for boto3 (or Spark `read.csv`/`write.mode("overwrite").partitionBy(...)`) changes the plumbing, not the design.

**Complexity:** O(bytes in selected partitions).

**Common mistakes**

- Appending to target partitions → duplicates on reruns.
- Committing the bookmark before outputs are written.